![imagenes](logo.png)

# Métodos de ensamblado

El ensemble learning consiste en combinar varios modelos débiles (o moderadamente buenos) para formar un modelo fuerte, con mejor rendimiento y menor varianza o sesgo.
La lógica es parecida a un comité de expertos: cada uno se equivoca de manera diferente, pero al promediar o votar, los errores tienden a cancelarse.

| Tipo de ensamblado                  | Estrategia                                                                                   | Ejemplo clásico                                          |
| ----------------------------------- | -------------------------------------------------------------------------------------------- | -------------------------------------------------------- |
| **Bagging** (Bootstrap Aggregating) | Entrenar varios modelos **en paralelo** sobre subconjuntos aleatorios de los datos           | Random Forest                                            |
| **Boosting**                        | Entrenar varios modelos **en secuencia**, donde cada modelo corrige los errores del anterior | AdaBoost, Gradient Boosting, XGBoost, LightGBM, CatBoost |


## Bagging

“Promediar para reducir varianza”.

1. Se generan m muestras bootstrap del conjunto de entrenamiento (es decir, se toman muestras con reemplazo del dataset original).

2. Se entrena un modelo independiente en cada una de esas muestras.

3. En clasificación: se hace votación mayoritaria. En regresión: se toma el promedio de las predicciones.

### Propósito

- Reducir la varianza (los árboles de decisión son inestables, pequeños cambios en los datos los alteran mucho).

- No afecta mucho al sesgo, pero disminuye el sobreajuste al suavizar las predicciones.

**Ejemplo.** El Random Forest es un caso particular de bagging con árboles.

### Random Forest

El Random Forest añade aleatoriedad adicional para que los árboles sean más diversos:

1. Cada árbol se entrena sobre una muestra bootstrap diferente.

2. En cada división (split) del árbol, no se consideran todas las variables, sino un subconjunto aleatorio de ellas.

Esto rompe la correlación entre árboles, mejorando la diversidad y por tanto la estabilidad del promedio.

**Características**

- Reducción de varianza aún mayor que el bagging puro.

- Generalmente muy buen desempeño out-of-the-box.

- Medidas de importancia de variables (feature importance).

- Posibilidad de estimar el error OOB (out-of-bag), usando los ejemplos no incluidos en cada muestra bootstrap.

**Desventajas**

- Menos interpretable que un solo árbol.

- Puede ser computacionalmente costoso si hay muchos árboles o variables.


## Boosting

“Aprender de los errores”.

- Se entrena un modelo base (por ejemplo, un árbol pequeño o stump).

- Se evalúan los errores y se aumenta el peso de las observaciones mal clasificadas.

- Se entrena el siguiente modelo concentrándose más en esos errores.

- Se combinan los modelos (en general, mediante una suma ponderada).

Así, cada nuevo modelo corrige el sesgo del anterior.

| Aspecto          | Bagging          | Boosting             |
| ---------------- | ---------------- | -------------------- |
| Estrategia       | Paralela         | Secuencial           |
| Objetivo         | Reducir varianza | Reducir sesgo        |
| Peso de muestras | Igual            | Cambia según errores |
| Modelos base     | Independientes   | Dependientes         |
| Ejemplo          | Random Forest    | AdaBoost, XGBoost    |



<center>
  <img src="im036.png" width="600" height="300">
</center>

### Ejemplos concretos de boosting

🔹 AdaBoost

- Los pesos de las observaciones cambian en función de si el modelo anterior las clasificó mal.

- Cada modelo tiene un peso según su precisión.

- La predicción final es una combinación ponderada de los modelos.

🔹 Gradient Boosting

- En vez de ajustar pesos de observaciones, ajusta el nuevo modelo para minimizar el error residual del modelo anterior.

- Se entiende como un descenso de gradiente en el espacio de funciones.

- Cada árbol intenta corregir los residuos (errores) del ensamble anterior.

🔹 XGBoost / LightGBM / CatBoost

- Versiones optimizadas del gradient boosting:

- XGBoost: regularización explícita (L1, L2), paralelización, manejo de missing values.

- LightGBM: usa histogramas y técnicas más rápidas de partición.

- CatBoost: diseñado para variables categóricas.

## Supongamos un caso simple

Queremos predecir si un punto pertenece a la clase 1 (sí) o 0 (no).

Tenemos dos modelos base:

- $M_1$: predice $\hat{y}_1 = 0.6$
- $M_2$: predice $\hat{y}_2 = 0.4$

En bagging, el resultado sería el promedio:

$$\hat{y}_{bag} = \frac{0.6 + 0.4}{2} = 0.5$$

(es decir, cada modelo cuenta igual).

En boosting, los modelos no pesan igual.

El segundo modelo se entrena para corregir los errores del primero, y el ensamble final se forma con pesos $(\alpha_1, \alpha_2)$ proporcionales a la precisión de cada uno.

## **Ejemplo estilo AdaBoost**

Supongamos que:
- $M_1$ tuvo precisión del 80%, así que le asignamos peso $\alpha_1 = 0.8$
- $M_2$ tuvo precisión del 60%, peso $\alpha_2 = 0.6$

El modelo final sería una combinación ponderada:

$$\hat{y}_{boost} = \frac{\alpha_1 \hat{y}_1 + \alpha_2 \hat{y}_2}{\alpha_1 + \alpha_2}$$

Sustituyendo:

$$\hat{y}_{boost} = \frac{(0.8)(0.6) + (0.6)(0.4)}{0.8 + 0.6} = \frac{0.48 + 0.24}{1.4} = 0.514$$

— Resultado aproximado: 0.5, que es lo que el diagrama simboliza.